### Import Dependencies

In [ ]:
from openai import OpenAI
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### Ticket to be classified

In [ ]:
ticket = "I was charged twice for my subscription and now I can't log in at all."

### OpenAI Decisions API

In [ ]:
openai_client = OpenAI()

#### Predicate

In [ ]:
response_predicate_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=ticket,
    questions=[
        {
            "type": "predicate",
            "name": "double_charge",
            "instructions": "The customer was charged more than once.",
        }
    ]
)

In [ ]:
response_predicate_oai.answers[0]

#### Choice

In [ ]:
response_choice_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=ticket,
    questions=[
        {
            "type": "choice",
            "name": "category",
            "instructions": "Classify the support ticket.",
            "choices": [
                {"value": "billing", "description": "Payments, charges, refunds"},
                {"value": "technical", "description": "Login problems, errors, outages"},
                {"value": "other", "description": "Anything else"},
            ],
        }
    ]
)

In [ ]:
response_choice_oai.answers[0]

In [ ]:
for probability in response_choice_oai.answers[0].probabilities:
    print(f"{probability.value}: {probability.probability}")

#### Score

In [ ]:
response_score_oai = openai_client.decisions.create(
    model="gpt-6-luna",
    input=ticket,
    questions=[
        {
            "type": "score",
            "name": "urgency",
            "instructions": "How urgent is this ticket?",
            "levels": [
                {"label": "Low"},
                {"label": "Medium"},
                {"label": "High"},
            ],
        }
    ]
)

In [ ]:
response_score_oai.answers[0]

In [ ]:
for probability in response_score_oai.answers[0].probabilities:
    print(f"{probability.value}: {probability.probability}")

In [ ]:
print(response_score_oai.answers[0].score)

### TypeSafeAI SDK

In [ ]:
typesafe_client = TypeSafeClient()

#### Noul

In [ ]:
response_noul_ts = typesafe_client.system_one(
    state=ticket,
    questions={
        "double_charge": Noul(
            instructions="The customer was charged more than once",
        )
    }
)

In [ ]:
response_noul_ts.answers["double_charge"].noul

#### Choice

In [ ]:
response_choice_ts = typesafe_client.system_one(
    state=ticket,
    questions={
        "category": Choice(
            instructions="Classify the support ticket",
            criteria={
                "billing": "Payments, charges, refunds",
                "technical": "Login problems, errors, outages",
                "other": "Anything else",
            }
        )
    },
)

In [ ]:
response_choice_ts.answers["category"].probabilities

#### Score

In [ ]:
response_score_ts = typesafe_client.system_one(
    state=ticket,
    questions={
        "urgency": Score(
            instructions="How urgent is this ticket",
            criteria=["Low", "Medium", "High"],
        )
    }
)

In [ ]:
response_score_ts.answers["urgency"].score